[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/02_multimodal_beyond_vision/02_multimodal_beyond_vision.ipynb)

# 02. Beyond Vision: Audio + Text + Video

**This notebook covers:**
- Audio representations (spectrograms, Whisper embeddings)
- Audio-Text models (CLAP)
- Video understanding (temporal dimension)
- ImageBind: one embedding space for 6 modalities
- Building an audio-image-text model from scratch

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/05_Advanced_Topics/02_multimodal_beyond_vision")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

![CLAP Architecture — Elizalde et al. (2023)](../assets/paper_figure_clap.png)

*Source: Elizalde et al. (2023) — "CLAP: Learning Audio Concepts from Natural Language Supervision" — [arXiv:2206.04769](https://arxiv.org/abs/2206.04769)*

## 🎵 Audio Self-Supervised Learning: wav2vec 2.0 & HuBERT

### wav2vec 2.0 (Baevski et al., 2020)
**Paper:** [arXiv:2006.11477](https://arxiv.org/abs/2006.11477)

wav2vec 2.0 learns speech representations by solving a **contrastive task** over masked audio segments:

**Pipeline:**
```
Raw Audio → CNN Feature Encoder → Quantization Module → Contrastive Loss
                    ↓                                          ↑
            Transformer Encoder → Contextualized Representations
                    ↓ (masked positions)
            Predict quantized targets
```

**Contrastive Loss:**

$$
\mathcal{L} = -\log \frac{\exp(\text{sim}(c_t, q_t) / \kappa)}{\sum_{\tilde{q} \in Q_t} \exp(\text{sim}(c_t, \tilde{q}) / \kappa)}
$$

where $c_t$ is the context output at masked position $t$, $q_t$ is the true quantized target, and $Q_t$ includes distractors.

**Diversity Loss** (prevents codebook collapse):

$$
\mathcal{L}_d = \frac{1}{GV} \sum_{g=1}^{G} -H(\bar{p}_g) = \frac{1}{GV} \sum_{g=1}^{G} \sum_{v=1}^{V} \bar{p}_{g,v} \log \bar{p}_{g,v}
$$

### HuBERT (Hsu et al., 2021)
**Paper:** [arXiv:2106.07447](https://arxiv.org/abs/2106.07447)

Uses **offline clustering** (k-means on MFCCs) to create pseudo-labels, then trains with masked prediction:

$$
\mathcal{L}_{HuBERT} = \sum_{t \in M} -\log p(c_t \mid h_t^L)
$$

where $M$ are masked positions, $c_t$ is the cluster assignment, $h_t^L$ is the final hidden state.

### Whisper (Radford et al., 2023)
**Paper:** [arXiv:2212.04356](https://arxiv.org/abs/2212.04356)

Unlike wav2vec/HuBERT, Whisper uses **supervised** training on 680K hours of labeled audio:
- Encoder-decoder Transformer (like BART for audio)
- Multitask: transcription, translation, language ID, timestamp prediction
- No self-supervised pretraining — just scale!

| Model | Params | WER (LibriSpeech test-clean) |
|-------|--------|------------------------------|
| wav2vec 2.0 Base | 95M | 3.4% |
| HuBERT Large | 317M | 2.6% |
| Whisper Large-v3 | 1.5B | 2.0% |

![wav2vec 2.0 Architecture — Baevski et al. (2020)](../assets/paper_figure_audio_ssl.png)

*Source: Baevski et al. (2020) — "wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations" — [arXiv:2006.11477](https://arxiv.org/abs/2006.11477)*

*See also: HuBERT (Hsu 2021), Whisper (Radford 2023)*

## 🎬 Video Understanding: Temporal Modeling

### TimeSformer (Bertasius et al., 2021)
**Paper:** [arXiv:2102.05095](https://arxiv.org/abs/2102.05095)

**Key idea:** Factorize space-time attention into separate spatial and temporal attention:

$$
\text{Attn}_{divided}(Q, K, V) = \text{Softmax}\left(\frac{Q_s K_s^T}{\sqrt{d}}\right)V_s + \text{Softmax}\left(\frac{Q_t K_t^T}{\sqrt{d}}\right)V_t
$$

Attention patterns compared:
| Pattern | Complexity | Accuracy (K400) |
|---------|-----------|-----------------|
| Space-only | $O(S^2)$ per frame | 77.9% |
| Joint space-time | $O((S \cdot T)^2)$ | 78.0% |
| **Divided space-time** | $O(S^2 + T^2)$ per token | **78.0%** |

Divided attention achieves the same accuracy as joint at **much lower cost!**

### Video-LLaVA (Lin et al., 2023)
**Paper:** [arXiv:2311.10122](https://arxiv.org/abs/2311.10122)

Extends LLaVA to video by:
1. Sample T frames uniformly
2. Encode each frame with ViT → get T × P patch tokens
3. Concatenate all frame tokens → feed into LLM
4. The LLM's self-attention handles temporal reasoning

### Segment Anything Model (SAM) (Kirillov et al., 2023)
**Paper:** [arXiv:2304.02643](https://arxiv.org/abs/2304.02643)

SAM is a **foundation model for image segmentation** — relevant to multimodal because it enables:
- **Visual grounding:** Given text → find region → SAM segments it
- **Interactive segmentation:** Points, boxes, or text as prompts
- Trained on **SA-1B** dataset (1 billion masks, 11M images)

Architecture:
```
Image Encoder (ViT-H) → Image Embedding
                              ↓
Prompt Encoder (points/boxes/text) → Prompt Tokens
                              ↓
Mask Decoder (lightweight) → Segmentation Masks
```

![ImageBind Overview — Girdhar et al. (2023)](../assets/paper_figure_imagebind.png)

*Source: Girdhar et al. (2023) — "ImageBind: One Embedding Space To Bind Them All" — [arXiv:2305.05665](https://arxiv.org/abs/2305.05665)*

In [ ]:
# The multimodal landscape beyond vision+text

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('The Full Multimodal Landscape', fontsize=18, fontweight='bold', pad=20)

modalities = [
    (2, 7, 'Image\n(pixels)', '#E74C3C'),
    (5, 7, 'Text\n(tokens)', '#3498DB'),
    (8, 7, 'Audio\n(waveform)', '#2ECC71'),
    (11, 7, 'Video\n(frames)', '#F39C12'),
    (3.5, 5.5, 'Depth\n(3D)', '#9B59B6'),
    (10, 5.5, 'IMU/Sensor\n(motion)', '#1ABC9C'),
]

for x, y, label, color in modalities:
    draw_architecture_block(ax, x, y, 2.5, 0.8, label, color)

draw_architecture_block(ax, 7, 3.5, 10, 1.2, 'Shared Embedding Space\n(e.g., ImageBind: ONE space for ALL modalities)', '#34495E', fontsize=11)

for x, y, _, _ in modalities:
    draw_arrow(ax, (x, y-0.5), (7 + (x-7)*0.3, 4.2))

tasks = [
    (3, 1.5, 'Cross-modal\nRetrieval'),
    (7, 1.5, 'Zero-shot\nClassification'),
    (11, 1.5, 'Any-to-Any\nGeneration'),
]
for x, y, label in tasks:
    draw_architecture_block(ax, x, y, 3, 0.7, label, '#7F8C8D')
    draw_arrow(ax, (x, 2.8), (x, 2.0))

plt.tight_layout()
plt.savefig('../assets/full_multimodal.png', dpi=150, bbox_inches='tight')
plt.show()

## 1. Audio Representations

## Audio Processing Math

Before a neural network can process audio, raw waveforms must be transformed into a representation that captures frequency content over time.

### Sampling

Audio is digitized by measuring amplitude at regular intervals. Standard speech models use **16 kHz** (16,000 samples per second):

$$x[n] = x(n / f_s), \quad n = 0, 1, 2, \ldots, \quad f_s = 16000 \text{ Hz}$$

**Nyquist theorem:** To capture frequencies up to $f_{\max}$, we must sample at $f_s \geq 2 f_{\max}$. At 16 kHz, the maximum representable frequency is 8 kHz — sufficient for human speech (fundamental frequency ~85–255 Hz, formants up to ~4 kHz).

### Short-Time Fourier Transform (STFT)

The STFT computes the frequency content in sliding windows over the signal:

$$X(t, f) = \sum_{n=0}^{N-1} x(n + tH) \cdot w(n) \cdot e^{-j2\pi fn/N}$$

where:
- $w(n)$ is a window function (typically Hann window) that tapers signal edges to reduce spectral leakage
- $H$ is the **hop length** (stride between windows) — smaller $H$ gives finer time resolution
- $N$ is the **window size** (FFT length) — larger $N$ gives finer frequency resolution

This produces a complex-valued time-frequency representation: $|X(t,f)|$ gives magnitude, $\angle X(t,f)$ gives phase.

### Mel Scale

Human hearing is **logarithmic** in frequency — we distinguish low frequencies more finely than high ones. The mel scale captures this:

$$m = 2595 \log_{10}\!\left(1 + \frac{f}{700}\right)$$

| Frequency (Hz) | Mel |
|----------------|-----|
| 0 | 0 |
| 500 | ~500 |
| 1000 | ~1000 |
| 4000 | ~2142 |
| 8000 | ~2840 |

### Mel Spectrogram

Apply a mel filterbank matrix $M \in \mathbb{R}^{n_{\text{mels}} \times n_{\text{freq}}}$ to the STFT magnitude squared:

$$S_{\text{mel}} = M \cdot |X|^2$$

Then apply log compression for numerical stability and perceptual scaling:

$$S_{\text{log-mel}} = \log(S_{\text{mel}} + \epsilon)$$

### Full Pipeline

$$\text{Waveform} \xrightarrow{\text{STFT}} |X|^2 \xrightarrow{\text{Mel filterbank}} S_{\text{mel}} \xrightarrow{\log} \text{Log-Mel Spectrogram} \xrightarrow{\text{Encoder}} \text{Embedding}$$

The resulting log-mel spectrogram is treated as a 2D "image" (frequency × time) and fed into a CNN or ViT encoder — the same architectures used for vision work surprisingly well for audio.

In [ ]:
# Visualize: How audio becomes embeddings

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Audio Processing Pipeline', fontsize=16, fontweight='bold')

# 1. Waveform
ax = axes[0]
t = np.linspace(0, 1, 1000)
waveform = np.sin(2 * np.pi * 440 * t) * np.exp(-2*t) + np.random.randn(1000) * 0.1
ax.plot(t, waveform, color='#2ECC71', linewidth=0.5)
ax.set_title('1. Raw Waveform')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')

# 2. Spectrogram
ax = axes[1]
spec = np.random.rand(64, 100) ** 2
# Add some structure
for f in [10, 20, 40]:
    spec[f-2:f+2, :] += np.random.rand(4, 100) * 2
ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
ax.set_title('2. Mel Spectrogram')
ax.set_xlabel('Time frames')
ax.set_ylabel('Mel frequency bins')

# 3. Patch embedding (like ViT for audio)
ax = axes[2]
patches = np.random.randn(16, 32) * 0.5
ax.imshow(patches, aspect='auto', cmap='RdBu_r')
ax.set_title('3. Patch Embeddings\n(spectrogram patches)')
ax.set_xlabel('Embedding dim (first 32)')
ax.set_ylabel('Patch index')

# 4. Output embedding
ax = axes[3]
embedding = np.random.randn(64)
ax.bar(range(64), embedding, color='#2ECC71', alpha=0.7)
ax.set_title('4. Audio Embedding\n(fixed-size vector)')
ax.set_xlabel('Dimension')
ax.set_ylabel('Value')

plt.tight_layout()
plt.savefig('../assets/audio_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build a simple audio encoder

class SimpleAudioEncoder(nn.Module):
    """Encodes a mel spectrogram into an embedding vector."""
    def __init__(self, n_mels=64, embed_dim=128, n_layers=2):
        super().__init__()
        # Treat spectrogram as 1-channel image
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, embed_dim, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.flatten_dim = embed_dim * 4 * 4
        self.proj = nn.Linear(self.flatten_dim, embed_dim)

    def forward(self, mel_spec):
        # mel_spec: [B, 1, n_mels, T]
        x = self.conv(mel_spec).flatten(1)
        return self.proj(x)


audio_enc = SimpleAudioEncoder(embed_dim=128)
dummy_mel = torch.randn(2, 1, 64, 100)  # batch=2, 64 mel bins, 100 time frames
audio_emb = audio_enc(dummy_mel)
print(f"Audio input: {dummy_mel.shape}")
print(f"Audio embedding: {audio_emb.shape}")
count_parameters(audio_enc)

In [ ]:
# ============================================================
#  Example: Audio Processing — STFT from Scratch
# ============================================================

print("=" * 65)
print("  AUDIO PROCESSING: STFT Step-by-Step")
print("=" * 65)

# Generate a synthetic audio signal (sum of sine waves)
sample_rate = 16000
duration = 1.0  # 1 second
t = torch.linspace(0, duration, int(sample_rate * duration))

# Create a signal with 3 frequencies: 440Hz (A4), 880Hz (A5), 220Hz (A3)
signal = (torch.sin(2 * np.pi * 440 * t) * 0.5 + 
          torch.sin(2 * np.pi * 880 * t) * 0.3 + 
          torch.sin(2 * np.pi * 220 * t) * 0.2)

# Add some noise
signal += torch.randn_like(signal) * 0.05

print(f"Audio signal: {len(signal)} samples at {sample_rate}Hz ({duration}s)")
print(f"Frequencies: 220Hz (A3), 440Hz (A4), 880Hz (A5)")
print(f"Nyquist frequency: {sample_rate/2}Hz")

# Compute STFT manually
n_fft = 512
hop_length = 160
window = torch.hann_window(n_fft)

# STFT
stft_result = torch.stft(signal, n_fft=n_fft, hop_length=hop_length, 
                          win_length=n_fft, window=window, return_complex=True)
magnitude = stft_result.abs()
phase = stft_result.angle()

print(f"\nSTFT parameters:")
print(f"  FFT size: {n_fft}")
print(f"  Hop length: {hop_length}")
print(f"  Window: Hann")
print(f"  Frequency bins: {magnitude.shape[0]} (0 to {sample_rate/2}Hz)")
print(f"  Time frames: {magnitude.shape[1]}")
print(f"  Frequency resolution: {sample_rate/n_fft:.1f}Hz per bin")
print(f"  Time resolution: {hop_length/sample_rate*1000:.1f}ms per frame")

# Convert to Mel scale (simplified)
n_mels = 80
mel_low = 0
mel_high = 2595 * np.log10(1 + (sample_rate/2) / 700)  # Hz to Mel
mel_points = torch.linspace(mel_low, mel_high, n_mels + 2)
hz_points = 700 * (10 ** (mel_points / 2595) - 1)  # Mel to Hz

# Create mel filterbank
freq_bins = torch.linspace(0, sample_rate/2, n_fft//2 + 1)
mel_filters = torch.zeros(n_mels, n_fft//2 + 1)
for i in range(n_mels):
    low = hz_points[i]
    center = hz_points[i + 1]
    high = hz_points[i + 2]
    
    for j, f in enumerate(freq_bins):
        if low <= f <= center:
            mel_filters[i, j] = (f - low) / (center - low + 1e-8)
        elif center < f <= high:
            mel_filters[i, j] = (high - f) / (high - center + 1e-8)

mel_spec = mel_filters @ magnitude
log_mel = torch.log(mel_spec + 1e-8)

print(f"\nMel spectrogram: {log_mel.shape} ({n_mels} mels × {log_mel.shape[1]} frames)")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Waveform
ax = axes[0, 0]
ax.plot(t[:3000].numpy(), signal[:3000].numpy(), color='#3498DB', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Waveform (first 187ms)', fontsize=12, fontweight='bold')

# STFT Spectrogram
ax = axes[0, 1]
im = ax.imshow(torch.log(magnitude + 1e-8).numpy(), aspect='auto', origin='lower', cmap='magma',
               extent=[0, duration, 0, sample_rate/2])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('STFT Spectrogram (log magnitude)', fontsize=12, fontweight='bold')
# Mark the three frequencies
for freq in [220, 440, 880]:
    ax.axhline(y=freq, color='white', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(duration * 0.02, freq + 30, f'{freq}Hz', color='white', fontsize=9)
plt.colorbar(im, ax=ax)

# Mel filterbank
ax = axes[1, 0]
for i in range(0, n_mels, 10):
    ax.plot(freq_bins.numpy(), mel_filters[i].numpy(), alpha=0.7)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Filter Response')
ax.set_title(f'Mel Filterbank ({n_mels} filters)\n(more filters at low freq)', fontsize=12, fontweight='bold')

# Mel spectrogram
ax = axes[1, 1]
im = ax.imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='magma')
ax.set_xlabel('Time Frame')
ax.set_ylabel('Mel Bin')
ax.set_title('Mel Spectrogram\n(input to audio encoder)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('../assets/audio_stft_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ This mel spectrogram is what Whisper/CLAP see as input")
print(f"   Shape [{n_mels}, {log_mel.shape[1]}] → audio encoder → embedding vector")

## CLAP (Contrastive Language-Audio Pretraining)

**CLAP** applies the same contrastive learning principle as CLIP, but for **audio-text pairs** instead of image-text pairs.

### Contrastive Loss

Given a batch of $N$ audio-text pairs, CLAP uses symmetric InfoNCE loss:

$$\mathcal{L}_{\text{CLAP}} = -\frac{1}{2N}\sum_{i=1}^{N}\left[\log\frac{e^{s_{ii}/\tau}}{\sum_{j=1}^{N} e^{s_{ij}/\tau}} + \log\frac{e^{s_{ii}/\tau}}{\sum_{j=1}^{N} e^{s_{ji}/\tau}}\right]$$

where $s_{ij} = \mathbf{a}_i^\top \mathbf{t}_j$ is the cosine similarity between audio embedding $\mathbf{a}_i$ and text embedding $\mathbf{t}_j$, and $\tau$ is a learned temperature parameter.

### Architecture

| Component | Model | Output |
|-----------|-------|--------|
| **Audio encoder** | HTS-AT (Hierarchical Token-Semantic Audio Transformer) | $\mathbf{a} \in \mathbb{R}^{d}$ |
| **Text encoder** | RoBERTa | $\mathbf{t} \in \mathbb{R}^{d}$ |

HTS-AT processes mel spectrograms with a hierarchical design: local attention over time-frequency patches, then global attention over the full spectrogram — analogous to how ViT processes image patches.

### Applications

- **Audio classification:** Zero-shot classify audio by comparing against text descriptions of classes
- **Text-to-audio retrieval:** Given a text query, find matching audio clips in a database
- **Audio captioning:** Generate text descriptions of audio content
- **Audio-text alignment:** Foundation for audio-language models like AudioLM, MusicLM

CLAP demonstrates that contrastive learning generalizes beyond vision — the same recipe (dual encoders + InfoNCE + large-scale paired data) works for any modality paired with text.

## 2. Three-Modal Model: Image + Text + Audio

In [ ]:
class ThreeModalCLIP(nn.Module):
    """CLIP-style model for Image + Text + Audio."""
    def __init__(self, embed_dim=128, proj_dim=64):
        super().__init__()
        # Image encoder (small CNN)
        self.img_enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # Text encoder
        self.txt_enc = nn.Sequential(
            nn.Embedding(1000, embed_dim),
        )
        # Audio encoder
        self.aud_enc = SimpleAudioEncoder(embed_dim=embed_dim)

        # Shared projection heads
        self.img_proj = nn.Linear(embed_dim, proj_dim)
        self.txt_proj = nn.Linear(embed_dim, proj_dim)
        self.aud_proj = nn.Linear(embed_dim, proj_dim)

        self.temperature = nn.Parameter(torch.ones(1) * 0.07)

    def encode(self, images=None, text_ids=None, audio=None):
        outputs = {}
        if images is not None:
            ie = F.normalize(self.img_proj(self.img_enc(images)), dim=-1)
            outputs['image'] = ie
        if text_ids is not None:
            te = self.txt_enc(text_ids).mean(dim=1)
            te = F.normalize(self.txt_proj(te), dim=-1)
            outputs['text'] = te
        if audio is not None:
            ae = F.normalize(self.aud_proj(self.aud_enc(audio)), dim=-1)
            outputs['audio'] = ae
        return outputs


model = ThreeModalCLIP()
count_parameters(model)

# Test all modalities
embs = model.encode(
    images=torch.randn(4, 3, 32, 32),
    text_ids=torch.randint(0, 1000, (4, 10)),
    audio=torch.randn(4, 1, 64, 100)
)
for name, emb in embs.items():
    print(f"{name:6s} embedding: {emb.shape}")

# Cross-modal similarity
sim_img_txt = (embs['image'] @ embs['text'].T).detach()
sim_img_aud = (embs['image'] @ embs['audio'].T).detach()
sim_txt_aud = (embs['text'] @ embs['audio'].T).detach()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sim, title in zip(axes, [sim_img_txt, sim_img_aud, sim_txt_aud],
                          ['Image ↔ Text', 'Image ↔ Audio', 'Text ↔ Audio']):
    ax.imshow(sim.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(title, fontsize=12, fontweight='bold')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=9)

fig.suptitle('Cross-Modal Similarities (untrained)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Video Understanding

Video = Image + **temporal dimension**. Key approaches:

| Approach | How | Example |
|----------|-----|---------|
| Frame sampling | Encode N frames independently | VideoCLIP |
| Temporal attention | Add time attention layers | TimeSformer |
| 3D convolution | Conv across space + time | C3D, SlowFast |
| Token merging | Merge frame tokens | Video-LLaVA |

## Spatial-Temporal Factorization

Full 3D self-attention over a video treats every spatiotemporal patch as a token. For a video with $T$ frames and $N$ patches per frame, this yields $T \cdot N$ tokens with attention complexity:

$$\mathcal{O}(T^2 \cdot N^2) \quad \text{— prohibitively expensive}$$

For example, 8 frames × 196 patches = 1568 tokens → $1568^2 \approx 2.5M$ attention operations per layer.

### TimeSformer: Divided Space-Time Attention

**TimeSformer** factorizes full 3D attention into two cheaper operations:

1. **Spatial attention** (within each frame): Each of the $T$ frames attends over its $N$ patches independently.

$$\mathcal{O}(T \cdot N^2)$$

2. **Temporal attention** (across frames): Each of the $N$ spatial locations attends across $T$ frames.

$$\mathcal{O}(N \cdot T^2)$$

**Total complexity:** $\mathcal{O}(T \cdot N^2 + N \cdot T^2)$ vs. $\mathcal{O}(T^2 N^2)$ for full 3D attention.

For $T=8, N=196$: factorized = $8 \times 196^2 + 196 \times 8^2 \approx 320K$ vs. full = $1568^2 \approx 2.5M$ — roughly **8× cheaper**.

### Alternative: Divided Space-Time Attention

Some architectures **alternate** spatial and temporal attention layers rather than applying both at every layer:

$$\text{Layer 1: Spatial} \to \text{Layer 2: Temporal} \to \text{Layer 3: Spatial} \to \cdots$$

This further reduces compute while maintaining the ability to capture both "what" (spatial) and "when" (temporal) information. Video-LLaVA extends the LLaVA pattern by sampling frames, encoding each with the vision encoder, and applying temporal pooling or attention before concatenating with text tokens.

In [ ]:
class SimpleVideoEncoder(nn.Module):
    """Encode video by sampling frames + temporal attention."""
    def __init__(self, embed_dim=128, n_frames=8):
        super().__init__()
        self.n_frames = n_frames
        # Per-frame encoder
        self.frame_enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # Temporal attention
        self.temporal_pos = nn.Embedding(n_frames, embed_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.temporal_attn = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, frames):
        """frames: [B, n_frames, C, H, W]"""
        B, T, C, H, W = frames.shape
        # Encode each frame
        frame_feats = self.frame_enc(frames.view(B*T, C, H, W))  # [B*T, D]
        frame_feats = frame_feats.view(B, T, -1)                  # [B, T, D]
        # Add temporal position
        pos = self.temporal_pos(torch.arange(T, device=frames.device))
        frame_feats = frame_feats + pos
        # Temporal attention
        output = self.temporal_attn(frame_feats)
        return self.norm(output.mean(dim=1))  # [B, D] pool over time


video_enc = SimpleVideoEncoder(embed_dim=128, n_frames=8)
dummy_video = torch.randn(2, 8, 3, 32, 32)  # 2 videos, 8 frames
video_emb = video_enc(dummy_video)
print(f"Video input:     {dummy_video.shape} (batch, frames, C, H, W)")
print(f"Video embedding: {video_emb.shape}")
count_parameters(video_enc)

## ImageBind Architecture

**ImageBind** (Meta, 2023) aligns **6 modalities** in a single embedding space using **images as the binding modality**:

$$\text{Modalities: } \{\text{Image, Text, Audio, Depth, Thermal, IMU}\}$$

### The Binding Insight

Training all pairwise alignments requires $\binom{6}{2} = 15$ contrastive losses — expensive and data-hungry. ImageBind's key insight: **only train each modality paired with images** (5 pairs). Because images co-occur naturally with other modalities in the real world (a photo of a scene also has depth, audio, thermal signature), the image encoder acts as a **hub** that indirectly aligns all other modalities.

$$\mathcal{L} = \sum_{m \in \{T, A, D, Th, IMU\}} \mathcal{L}_{\text{InfoNCE}}(I, m)$$

Only **5 losses** instead of 15 — yet emergent zero-shot cross-modal retrieval works for pairs never explicitly trained, e.g., **audio ↔ text** without ever seeing audio-text pairs!

### Why This Works

Each non-image modality encoder learns to map its input into the **same space as the image encoder**. Since text is also aligned to images, text and audio end up in the same space *transitively*:

$$\text{Audio} \xrightarrow{\mathcal{L}(I,A)} \text{Image Space} \xleftarrow{\mathcal{L}(I,T)} \text{Text}$$

$$\Rightarrow \text{Audio} \approx \text{Text} \quad \text{(emergent zero-shot alignment)}$$

### Architecture per Modality

| Modality | Encoder | Input |
|----------|---------|-------|
| Image | ViT-H/14 | RGB pixels |
| Text | Transformer | Token sequence |
| Audio | HTS-AT | Mel spectrogram |
| Depth | ViT-H/14 | Depth map (treated as 1-channel image) |
| Thermal | ViT-H/14 | Thermal image |
| IMU | 1D conv + Transformer | 6-axis sensor readings |

All encoders output embeddings in $\mathbb{R}^{1024}$, L2-normalized, trained with InfoNCE against the image embedding of paired data.

![ImageBind Overview — Girdhar et al. (2023)](../../assets/paper_figures/imagebind_overview.png)
*Source: Girdhar et al. (2023) — ImageBind: One Embedding Space To Bind Them All — [arXiv:2305.05665](https://arxiv.org/abs/2305.05665)*

## 🧠 Multimodal Chain-of-Thought Reasoning

![Multimodal Chain-of-Thought — Zhang et al. (2023)](../../assets/paper_figures/mm_cot.png)
*Source: Zhang et al. (2023) — Multimodal Chain-of-Thought Reasoning in Language Models — [arXiv:2302.00923](https://arxiv.org/abs/2302.00923)*

Traditional CoT works for text. **Multimodal CoT** extends this to vision+language:

**Two-stage framework:**
1. **Rationale Generation:** Given image + question → generate reasoning steps
2. **Answer Inference:** Given image + question + rationale → predict answer

This achieves **91.68%** on ScienceQA (vs. GPT-3.5 at 75.17%), showing that explicit reasoning over visual content significantly helps.

### Visual Programming (VisProg)

**Paper:** Gupta & Kembhavi (2023) — *Visual Programming: Compositional visual reasoning without training* — [arXiv:2211.11559](https://arxiv.org/abs/2211.11559)

Instead of end-to-end models, VisProg uses LLMs to **write programs** that compose vision modules:

```
Query: "What is the breed of the dog sitting to the right of the cat?"
Program:
  DOG = LOC(image, "dog")
  CAT = LOC(image, "cat")  
  DOG_RIGHT = FILTER(DOG, "right of", CAT)
  RESULT = VQA(image, DOG_RIGHT, "What breed?")
```

## 🔍 Multimodal Hallucination

A critical challenge: MLLMs sometimes **hallucinate** — generating descriptions of objects or attributes that don't exist in the image.

### Types of Hallucination

| Type | Example | Paper |
|------|---------|-------|
| **Object Hallucination** | "There is a cat in the image" (no cat exists) | POPE (Li et al., 2023) |
| **Attribute Hallucination** | "The red car" (car is blue) | GAVIE (Liu et al., 2023) |
| **Relation Hallucination** | "The cup is on the table" (cup is beside table) | HallusionBench (Guan et al., 2023) |

### Mitigation Approaches

1. **RLHF-V** (Yu et al., 2024) — Fine-grained human feedback for hallucination reduction
2. **LLaVA-RLHF** (Sun et al., 2023) — RLHF specifically for vision-language alignment
3. **Woodpecker** (Yin et al., 2023) — Post-hoc correction using tool-augmented models
4. **VCD** (Leng et al., 2024) — Visual Contrastive Decoding to reduce hallucination

## 📊 Evaluation Benchmarks for MLLMs

| Benchmark | Focus | # Samples | Key Metric |
|-----------|-------|-----------|------------|
| **MME** (Fu et al., 2023) | Comprehensive MLLM eval | 14 subtasks | Accuracy |
| **MMMU** (Yue et al., 2024) | College-level multimodal | 11.5K | Accuracy |
| **MM-Bench** (Liu et al., 2023) | Systematic capability eval | 3K | Accuracy |
| **SEED-Bench** (Li et al., 2023) | 12 dimensions of comprehension | 19K | Accuracy |
| **Video-MME** (Fu et al., 2024) | Video understanding eval | 900 videos | Accuracy |
| **HallusionBench** (Guan et al., 2023) | Hallucination detection | 1.1K | Accuracy |
| **MathVista** (Lu et al., 2024) | Mathematical reasoning | 6.1K | Accuracy |
| **ChartQA** (Masry et al., 2022) | Chart understanding | 32K | Accuracy |
| **DocVQA** (Mathew et al., 2021) | Document understanding | 50K | ANLS |

## 🎯 Multimodal In-Context Learning

**Paper:** Alayrac et al. (2022) — Flamingo demonstrated that MLLMs can learn from few examples:

```
[Image1] → "A red sports car parked on the street"
[Image2] → "A white sedan driving on the highway"  
[Image3] → ???
```

The model learns the pattern from examples and generates appropriate descriptions!

### Key ICL Models

| Model | Max Shots | Method |
|-------|-----------|--------|
| **Flamingo** | 32 | Gated cross-attention + interleaving |
| **MMICL** | 8 | Multi-modal context compression |
| **Emu** | 16 | Predict-next-in-context training |
| **CoBSAT** | - | Concept bottleneck for ICL |

## Key Takeaways

1. **Audio** → mel spectrogram → treat like an image → encode with ViT/CNN
2. **Video** → sample frames → encode each → temporal attention
3. **ImageBind** proved you can align 6 modalities in ONE space using image as the anchor
4. The pattern is always: **Modality Encoder → Projection → Shared Space**
5. **Contrastive learning** works across any pair of modalities

---
**Next:** `03_efficient_deployment.ipynb` — Ship your model efficiently

---

## 📚 References & Further Reading

### Papers — Audio
- **Robust Speech Recognition via Large-Scale Weak Supervision (Whisper)** — Radford et al., 2022 — [arXiv:2212.04356](https://arxiv.org/abs/2212.04356) — Audio-to-text foundation model
- **CLAP: Learning Audio Concepts from Natural Language Supervision** — Elizalde et al., 2023 — [arXiv:2206.04769](https://arxiv.org/abs/2206.04769) — CLIP for audio
- **wav2vec 2.0: A Framework for Self-Supervised Learning of Speech** — Baevski et al., 2020 — [arXiv:2006.11477](https://arxiv.org/abs/2006.11477) — Self-supervised audio encoder
- **AudioLM: A Language Modeling Approach to Audio Generation** — Borsos et al., 2023 — [arXiv:2209.03143](https://arxiv.org/abs/2209.03143) — Audio generation

### Papers — Video
- **Is Space-Time Attention All You Need for Video Understanding? (TimeSformer)** — Bertasius et al., 2021 — [arXiv:2102.05095](https://arxiv.org/abs/2102.05095) — Factorized spatial-temporal attention
- **ViViT: A Video Vision Transformer** — Arnab et al., 2021 — [arXiv:2103.15691](https://arxiv.org/abs/2103.15691) — Video Transformer variants
- **Video-LLaVA: Learning United Visual Representation** — Lin et al., 2023 — [arXiv:2311.10122](https://arxiv.org/abs/2311.10122) — LLaVA for video understanding

### Papers — Unified Multi-Modal
- **ImageBind: One Embedding Space To Bind Them All** — Girdhar et al., 2023 — [arXiv:2305.05665](https://arxiv.org/abs/2305.05665) — 6-modality unified embeddings
- **Meta-Transformer: A Unified Framework for Multimodal Learning** — Zhang et al., 2023 — [arXiv:2307.10802](https://arxiv.org/abs/2307.10802) — 12 modalities, one model
- **Unified-IO 2: Scaling Autoregressive Multimodal Models** — Lu et al., 2024 — [arXiv:2312.17172](https://arxiv.org/abs/2312.17172) — Any-to-any generation

### Blog Posts & Cheat Sheets
- 🔗 [Whisper on Hugging Face](https://huggingface.co/openai/whisper-large-v3) — Use Whisper in one function call
- 🔗 [Librosa Documentation](https://librosa.org/doc/latest/) — Python audio analysis library
- 🔗 [ImageBind Demo](https://imagebind.metademolab.com/) — Interactive cross-modal retrieval
- 🔗 [Papers With Code: Video Understanding](https://paperswithcode.com/task/video-understanding) — Benchmarks and methods
- 🔗 [Lilian Weng: Generative Modeling for Audio](https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/) — Audio ML overview
- 🔗 [LAION CLAP](https://github.com/LAION-AI/CLAP) — Open-source CLAP implementation